In [1]:
import os
import torch
from tqdm import tqdm
import re
from datasets import load_dataset
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.metrics import classification_report, f1_score, mean_absolute_error
import json
import re
from transformers import AutoTokenizer

In [ ]:
# --------------------------------------------------------
# CONFIG
# --------------------------------------------------------
DATA_DIR = "../data/pt/"
cache_dir="../models/"
EMBED_KEY = "hidden_states_last_layer"
os.environ["HF_TOKEN"] = "..."

In [5]:
def get_target_index(length, percent):
    """Helper to calculate zero-based index from percentage (0.0 to 1.0)."""
    if length <= 0:
        return 0
    percent = max(0.0, min(1.0, percent))
    return int(percent * (length - 1))

def get_reasoning_length(data, tokenizer, model_name):
    """
    Helper to calculate the number of tokens in the reasoning/thinking phase 
    based on the model type and data structure.
    """
    reasoning_tokens = 0

    # Logic extracted from your original snippet
    if model_name in ["openai-gpt-oss-20b", "openai-gpt-oss-120b"]:
        text_content = data['parsed_outputs'][0].content[0].text
        # Construct the specific prompt structure for counting
        full_text = "<|channel|>analysis<|message|>" + text_content + "<|end|>"
        reasoning_tokens = tokenizer(
            full_text,
            return_tensors="pt",
            add_special_tokens=False
        )['input_ids'].shape[1]

    elif model_name in ["google-gemma-3-27b-it", "google-gemma-3-12b-it", "google-gemma-3-4b-it", "google-gemma-3-1b-it", "meta-llama-Llama-3.1-8B-Instruct", "meta-llama-Llama-3.1-70B-Instruct"]:
        # Split by review tag
        text_segment = data['parsed_outputs'].split("<REVIEW>")[0]
        reasoning_tokens = tokenizer(
            text_segment,
            return_tensors="pt",
            add_special_tokens=False
        )['input_ids'].shape[1]

    elif model_name in ["Qwen-Qwen3-32B", "Qwen-Qwen3-14B", "Qwen-Qwen3-8B", "Qwen-Qwen3-4B"]:
        if EMBED_KEY not in data:
            text_segment = data['parsed_outputs'].split("<REVIEW>")[0]
        else:
            text_segment = data['parsed_outputs']['think_tokens']
        reasoning_tokens = tokenizer(
            text_segment,
            return_tensors="pt",
            add_special_tokens=False
        )['input_ids'].shape[1]
        
    return reasoning_tokens
    
def calculate_detailed_token_stats(approach_name: str, model_name: str, tokenizer):
    """
    Analyzes files for a specific approach/model and returns the 
    average, min, and max token lengths for both reasoning and response.
    """
    print(f"Analyzing Token Distributions for: {model_name} ({approach_name})")

    all_files = sorted([
        f for f in os.listdir(DATA_DIR)
        if f.endswith(".pt")
    ])

    # Pattern to match both train and test files for this specific model/approach
    pattern = re.compile(rf"{re.escape(approach_name)}_(train|test)_{re.escape(model_name)}_\d+\.pt")
    
    reasoning_counts = []
    response_counts = []

    for fname in tqdm(all_files, desc="Calculating lengths"):
        if pattern.match(fname):
            path = os.path.join(DATA_DIR, fname)
            try:
                # Load with weights_only=False as you are dealing with potentially complex dicts
                data = torch.load(path, map_location="cpu", weights_only=False)
       
                if EMBED_KEY not in data:
                    if model_name in ["openai-gpt-oss-20b", "openai-gpt-oss-120b"]:
                        text_content = data['parsed_outputs']
                        # Construct the specific prompt structure for counting
                        full_text = "<|channel|>final<|message|>" + text_content + "<|end|>"
                        total_tokens = tokenizer(
                            full_text,
                            return_tensors="pt",
                            add_special_tokens=False
                        )['input_ids'].shape[1]
                
                    elif model_name in ["google-gemma-3-27b-it", "google-gemma-3-12b-it", "google-gemma-3-4b-it", "google-gemma-3-1b-it", "meta-llama-Llama-3.1-8B-Instruct", "meta-llama-Llama-3.1-70B-Instruct", "Qwen-Qwen3-32B", "Qwen-Qwen3-14B", "Qwen-Qwen3-8B", "Qwen-Qwen3-4B"]:
                        # Split by review tag
                        text_segment = data['parsed_outputs']
                        total_tokens = tokenizer(
                            text_segment,
                            return_tensors="pt",
                            add_special_tokens=False
                        )['input_ids'].shape[1]
                
                else:
                    total_tokens = len(data[EMBED_KEY])
                
                # Use your existing logic to find the boundary
                if model_name in ["openai-gpt-oss-20b", "openai-gpt-oss-120b"]:
                    reasoning_len = 0
                else:
                    reasoning_len = get_reasoning_length(data, tokenizer, model_name)
                
                # Ensure reasoning doesn't exceed total tokens recorded
                reasoning_len = min(reasoning_len, total_tokens)
                response_len = total_tokens - reasoning_len
                
                reasoning_counts.append(reasoning_len)
                response_counts.append(response_len)
                
                del data # Explicitly clear memory
            except Exception as e:
                print(f"Error skipping {fname}: {e}")

    if not reasoning_counts:
        print("No matching data found.")
        return None

    # Aggregate Statistics
    stats = {
        "reasoning": {
            "avg": sum(reasoning_counts) / len(reasoning_counts),
            "min": min(reasoning_counts),
            "max": max(reasoning_counts)
        },
        "response": {
            "avg": sum(response_counts) / len(response_counts),
            "min": min(response_counts),
            "max": max(response_counts)
        },
        "total_samples": len(reasoning_counts)
    }

    # Print a clean summary table
    print(f"\n{'='*45}")
    print(f" TOKEN STATISTICS: {model_name}")
    print(f" Approach: {approach_name} | Samples: {stats['total_samples']}")
    print(f"{'-'*45}")
    print(f" PHASE     |  AVG    |  MIN  |  MAX")
    print(f"{'-'*45}")
    print(f" Reasoning | {stats['reasoning']['avg']:>7.2f} | {stats['reasoning']['min']:>5} | {stats['reasoning']['max']:>5}")
    print(f" Response  | {stats['response']['avg']:>7.2f} | {stats['response']['min']:>5} | {stats['response']['max']:>5}")
    print(f"{'='*45}\n")

    return stats

In [6]:
# Example usage:
#model_id = "meta-llama/Llama-3.1-70B-Instruct"
#model_id = "Qwen/Qwen3-4B"
model_id = "openai/gpt-oss-20b"
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True, device_map='cpu')
stats = calculate_detailed_token_stats(approach_name="TPR", model_name=model_id.replace("/","-"), tokenizer=tokenizer)
stats

Analyzing Token Distributions for: openai-gpt-oss-20b (TPR)


Calculating lengths: 100%|██████████| 46802/46802 [00:00<00:00, 141139.24it/s]


 TOKEN STATISTICS: openai-gpt-oss-20b
 Approach: TPR | Samples: 277
---------------------------------------------
 PHASE     |  AVG    |  MIN  |  MAX
---------------------------------------------
 Reasoning |    0.00 |     0 |     0
 Response  |  126.13 |    21 |   231



{'reasoning': {'avg': 0.0, 'min': 0, 'max': 0},
 'response': {'avg': 126.12635379061372, 'min': 21, 'max': 231},
 'total_samples': 277}